# Creating nice figures

In this notebook we show an example of loading GEE data, extracting the data into an [xarray.Dataset](https://docs.xarray.dev/en/stable/getting-started-guide/quick-overview.html), and plotting a map and time series. This is intended to guide you in developing nice figures for reports and presentations, rather than relying on screenshots of the interactive plots that geemap produces.

We will rely on a packages called [wxee](https://wxee.readthedocs.io/en/latest/index.html), which provides methods to convert a GEE Image or ImageCollection into an xarray object.

## Load packages

Import Python packages that are used for the analysis.


In [ ]:
%matplotlib inline

import ee
import eemont
import wxee
import geemap as gmap
import contextily as ctx
import geemap.colormaps as cm
import matplotlib.pyplot as plt

### Connect to Google Earth Engine (GEE)

In [ ]:
ee.Authenticate()

***

## Load MODIS NDVI ImageCollection over a catchment in the MDB

In [ ]:
Map = gmap.Map(center=[-30.3, 150.1244], zoom=7)

#set the catchment ID to select a sub-basin (this is in the MDB)
catchment_id = 5040070700

#start and end dates of the analysis
start = '2021-01-01'
end = '2021-12-31'

#load the basins dataset and grab a subcatchment
all_basins = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_4")
sub_basin = all_basins.filter(ee.Filter.eq('HYBAS_ID', catchment_id))

#load the MODIS NDVI product
ndvi = (ee.ImageCollection('MODIS/061/MOD13A1')
                  .filterBounds(sub_basin) # filter to MDB
                  .filterDate(start, end) #timeseries
                  .preprocess() #this will do the rescaling
                  .select('NDVI') #just select the NDVI band
                  .map(lambda image: image.clip(sub_basin)) #clip the data to MDB
     )

#define a visualization dict
NDVI_vis = {'min':0.1, 'max':0.9,
           'palette':list(cm.palettes.ndvi),
           }

# Plot the first time-step
Map.addLayer(ndvi.first(), NDVI_vis, 'NDVI')
Map.add_colorbar(NDVI_vis['palette'])
Map

## Convert to xarray and create a nice figure

This brings the data into memory on our side, so we can't do this with a lot of data. We'll bring the data in a 1 km resolution to keep it low.

In [ ]:
ndvi = ndvi.wx.to_xarray(region=sub_basin.geometry().bounds(), scale=1_000, crs="EPSG:3577")
ndvi

### Resample the NDVI data into monthly means and annual mean map

In [ ]:
ndvi_monthly = ndvi['NDVI'].resample(time='MS').mean()

ndvi_annual = ndvi['NDVI'].mean('time')

## Convert the FeatureCollection into a geopandas dataframe

This will alllow us to plot it on the map same as the xarray dataset

In [ ]:
basin_gdf = ee.data.computeFeatures({
    'expression': sub_basin,
    'fileFormat': 'GEOPANDAS_GEODATAFRAME'
})

# Need to set the CRS.
# Make sure it matches the CRS of FeatureCollection geometries.
basin_gdf.crs = 'EPSG:4326'

#now convert to the same CRS we set for the xarray dataset
basin_gdf = basin_gdf.to_crs('EPSG:3577')

## Create a multipanel figure

We will show the annual average NDVI for the catchment, along with the annual time series of NDVI.

In [ ]:
#create a figure object
fig, (ax_map, ax_ts) = plt.subplots(
    1, 2,                          # 1 row, 2 columns
    figsize=(15, 4),              # Overall figure size
    gridspec_kw={'width_ratios': [1, 2.5]},  # Adjust panel width ratio
)

# add the map to the left side of the figure.
im = ndvi_annual.plot(ax=ax_map, cmap='gist_earth', add_labels=False, robust=True, add_colorbar=False)

# Manually add a colorbar, this allows us to control the size
cbar = plt.colorbar(im, ax=ax_map, shrink=0.7)
cbar.ax.set_title('NDVI', fontsize=8)

# We can also add a 'basemap' which can provide context for the raster image
ctx.add_basemap(ax_map, source=ctx.providers.CartoDB.VoyagerNoLabels, crs='EPSG:3577', attribution='', attribution_size=1)

# lets also add the outline of our catchment to the map
basin_gdf.plot(ax=ax_map,
    facecolor="none",  # Transparent fill
    edgecolor="black",  # Black border
    linewidth=1 )

# add a title
ax_map.set_title('Annual mean NDVI');

#remove the lat/lon tick labels for tidiness
ax_map.set_yticklabels([])
ax_map.set_xticklabels([])

# Now add the NDVI timeseries to the plot, average out the spatial dims so we get a 1D timeseries
ndvi_monthly.mean(['x', 'y']).plot(ax=ax_ts)
ax_ts.grid(alpha=0.75)
ax_ts.set_title('NDVI averaged over the catchment');
ax_ts.set_xlabel(None);